# Generators overview

**A tour of every generator.** This notebook draws a few samples from each
structure generator in `compgen.gen` with fixed example parameters, first in 2D
and then in 3D, so you can see at a glance what each one produces.

To study a single generator in detail (your own parameters, a seed you can
change, an interactive 3D view), use `generator_viewer.ipynb` instead.

Every generator returns a boolean array of shape `N` (`True`/`False` are the two
phases). Parameters given as a `[min, max]` list are sampled uniformly on each
draw, so iterating a generator gives a family of related structures.

`Reconstruction2D` has an optional dependency, MCRpy
(`pip install compgen[mcrpy]`); that cell prints a message and moves on if it is
missing.

In [ ]:
from itertools import islice

import matplotlib.pyplot as plt
import numpy as np
from skimage import measure

from compgen import gen, utils

SEED = 42
N2 = 128  # 2D resolution
N3 = 48   # 3D resolution

In [ ]:
def show_2d(generator, n=4, title=None):
    """Draw n samples from a 2D generator and plot them side by side."""
    samples = list(islice(generator, n))
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    for ax, img in zip(np.atleast_1d(axes), samples):
        ax.imshow(img.T, cmap="gray", origin="lower", interpolation="nearest")
        ax.set_title(f"vf = {img.mean():.2f}", fontsize=9)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return samples


def show_3d(volume, title=None, phase=None):
    """Plot a 3D structure: the surface of one phase plus three mid-plane slices.

    `phase` selects which phase (True/False) is rendered as a surface; by
    default the minority phase is shown. The volume is padded with the other
    phase before marching cubes so the rendered phase appears as closed solids
    cut by the unit-cell faces.
    """
    if phase is None:
        phase = volume.mean() <= 0.5
    solid = volume if phase else ~volume
    padded = np.pad(solid, 1, constant_values=False).astype(float)
    verts, faces, _, _ = measure.marching_cubes(padded, 0.5)
    verts = (verts - 1) / np.array(volume.shape)

    fig = plt.figure(figsize=(13, 3.6))
    ax = fig.add_subplot(1, 4, 1, projection="3d")
    ax.plot_trisurf(
        verts[:, 0], verts[:, 1], faces, verts[:, 2],
        color="tab:blue", linewidth=0, antialiased=False, shade=True,
    )
    ax.set(xlim=(0, 1), ylim=(0, 1), zlim=(0, 1), xticks=[], yticks=[], zticks=[])
    ax.set_box_aspect((1, 1, 1))
    ax.set_title(f"vf = {volume.mean():.2f}", fontsize=9)

    mid = [s // 2 for s in volume.shape]
    slices = {
        "x = 1/2": volume[mid[0], :, :],
        "y = 1/2": volume[:, mid[1], :],
        "z = 1/2": volume[:, :, mid[2]],
    }
    for k, (name, sl) in enumerate(slices.items(), start=2):
        ax2 = fig.add_subplot(1, 4, k)
        ax2.imshow(sl.T, cmap="gray", origin="lower", interpolation="nearest")
        ax2.set_title(name, fontsize=9)
        ax2.axis("off")
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def show_3d_samples(generator, n=2, title=None, phase=None):
    """Draw n samples from a 3D generator and plot each with show_3d."""
    for i, vol in enumerate(islice(generator, n)):
        show_3d(vol, title=f"{title} — sample {i + 1}" if title else None, phase=phase)

## 2D structures

### Random

Uncorrelated noise at the requested volume fraction (here sampled from a range).

In [ ]:
show_2d(gen.RandomGenerator([N2, N2], vf=[0.3, 0.6], random_seed=SEED), title="RandomGenerator");

### Spinodal

A Gaussian random field band-passed around wavelength `l`, thresholded at the
target volume fraction. `l` and `bandwidth` are in pixels.

In [ ]:
show_2d(
    gen.SpinodalGenerator([N2, N2], vf=[0.3, 0.6], l=[60, 120], bandwidth=[2, 5], random_seed=SEED),
    title="SpinodalGenerator",
);

### Two-point statistics

Fits a covariance model to the two-point correlation of a reference image (here a
segmented micrograph of a real composite) and samples new structures with the
same statistics.

In [ ]:
reference = utils.load_and_segment("composite.jpg")
side = min(reference.shape)
reference = reference[:side, :side]

plt.figure(figsize=(3, 3))
plt.imshow(reference.T, cmap="gray", origin="lower")
plt.title("reference")
plt.axis("off")
plt.show()

show_2d(
    gen.TwoPointGenerator([N2, N2], reference_image=reference, radial_mask=50, random_seed=SEED),
    title="TwoPointGenerator",
);

### Voronoi grains

Periodic Voronoi tessellation with thick grain boundaries. `grain_size` and
`thickness` are fractions of the unit cell and of the grain size respectively;
`lloyd_iter` relaxes the seeds toward more equiaxed grains. `invert=False` keeps
the convention grains `True` / boundaries `False`.

In [ ]:
show_2d(
    gen.VoronoiGenerator([N2, N2], grain_size=[0.12, 0.25], thickness=0.15, invert=False, random_seed=SEED),
    title="VoronoiGenerator (Poisson seeds)",
)
show_2d(
    gen.VoronoiGenerator([N2, N2], grain_size=[0.12, 0.25], thickness=0.15, lloyd_iter=5, invert=False, random_seed=SEED),
    title="VoronoiGenerator (lloyd_iter=5)",
);

### Particles

Randomly placed disks, ellipses and laminae. Passing `buffer` enforces an
(approximate) minimum clearance between particles via overlap relaxation.

In [ ]:
particles_2d = [
    ("disks", {"size_m": 0.12, "size_s": 0.3}, None),
    ("disks", {"size_m": 0.1}, 0.01),
    ("ellipses", {"size_m": 0.2, "ratio_m": 3.0, "ratio_s": 1.0}, None),
    ("ellipses", {"size_m": 0.25, "ratio_m": 4.0, "theta_m": 30.0, "theta_s": 10.0}, None),
    ("laminae2", {"size_m": 0.08, "size_s": 0.3}, None),
]
for kind, params, buffer in particles_2d:
    show_2d(
        gen.ParticleGenerator([N2, N2], vf=0.3, kind=kind, shape_params=params, buffer=buffer, random_seed=SEED),
        title=f"ParticleGenerator: {kind}, {params}" + (f", buffer={buffer}" if buffer else ""),
    )

### Statistical reconstruction (MCRpy)

Reconstructs new structures matching the descriptors of a target image. This is
much slower than the other generators, so only a couple of small samples are drawn.

In [ ]:
try:
    show_2d(
        gen.Reconstruction2D([64, 64], "complete/reconstruction/demo_particles_35.npy", random_seed=SEED),
        n=2,
        title="Reconstruction2D",
    )
except RuntimeError as err:  # MCRpy not installed
    print(err)

### From a NumPy file

`FromNumpyGenerator` replays structures stored in a `.npy` stack (shape
`(n, nx, ny[, nz])`), e.g. to mesh/simulate a fixed set of images.

In [ ]:
show_2d(gen.FromNumpyGenerator("test_structures.npy"), n=1, title="FromNumpyGenerator");

## 3D structures

Each 3D sample is shown as the surface of the minority phase (left) and three mid-plane
slices through the unit cell.

### Random

In [ ]:
show_3d_samples(gen.RandomGenerator([N3] * 3, vf=0.3, random_seed=SEED), n=1, title="RandomGenerator");

### Spinodal

In [ ]:
show_3d_samples(
    gen.SpinodalGenerator([N3] * 3, vf=[0.3, 0.5], l=[20, 40], bandwidth=[2, 4], random_seed=SEED),
    title="SpinodalGenerator",
);

### Voronoi: walls on the cell faces

`mode="faces"` (the default) puts a wall of uniform full width
`thickness * grain_size` on every grain boundary (Voronoi face).

In [ ]:
show_3d_samples(
    gen.VoronoiGenerator([N3] * 3, grain_size=0.35, thickness=0.12, mode="faces", invert=False, random_seed=SEED),
    title="VoronoiGenerator, mode='faces'",
);

### Voronoi: struts on the cell edges

`mode="edges"` puts a cylinder of diameter `thickness * grain_size` along every
Voronoi edge, giving an open-cell foam/lattice-like network.

In [ ]:
show_3d_samples(
    gen.VoronoiGenerator([N3] * 3, grain_size=0.35, thickness=0.25, mode="edges", lloyd_iter=3, invert=False, random_seed=SEED),
    title="VoronoiGenerator, mode='edges'",
);

### Particles

Spheres, spheroids, fibers and laminae.

In [ ]:
particles_3d = [
    ("spheres", {"size_m": 0.2, "size_s": 0.3}),
    ("spheroids", {"size_m": 0.2, "ratio_m": 2.5}),
    ("fibers", {"size_m": 0.08, "ratio_m": 8.0}),
    ("laminae3", {"size_m": 0.08}),
]
for kind, params in particles_3d:
    show_3d_samples(
        gen.ParticleGenerator([N3] * 3, vf=0.2, kind=kind, shape_params=params, random_seed=SEED),
        n=1,
        title=f"ParticleGenerator: {kind}",
    )